# House Price Prediction — Feature Engineering

This notebook demonstrates the feature-engineering techniques relevant
to our House Price Prediction project, using the California Housing
dataset loaded directly with `sklearn.datasets.fetch_california_housing`.

It covers:
- The difference between numerical and categorical features.
- One-Hot Encoding and Label Encoding, demonstrated on a small
  illustrative example (since this dataset itself has no categorical
  columns — more on that below).
- Feature scaling with `StandardScaler`.
- Feature selection by correlation with the target.
- Data leakage, and why a transformer must be fit on training data
  only.

**This notebook does not train any regression model, and does not
build the Streamlit dashboard** — it's feature-engineering technique
demonstration only.


## 1. Load the Dataset

We load the dataset with `fetch_california_housing(as_frame=True)`,
which returns a scikit-learn `Bunch` whose `.frame` is a single
DataFrame containing the 8 feature columns plus the target,
`MedHouseVal`.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

housing_bunch = fetch_california_housing(as_frame=True)
housing_df = housing_bunch.frame

print(f"Dataset loaded: {housing_df.shape[0]} rows, {housing_df.shape[1]} columns.")
housing_df.head()


Dataset loaded: 20640 rows, 9 columns.


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Separate Features and Target

Almost every technique in this notebook works on the FEATURES (`X`)
only — the target (`y`) is what we're trying to predict, not
something we'd scale, encode, or otherwise transform as an input.


In [ ]:
X = housing_df.drop(columns=["MedHouseVal"])
y = housing_df["MedHouseVal"]

print("X shape (features):", X.shape)
print("y shape (target):  ", y.shape)
print("\nFeature columns:", list(X.columns))


X shape (features): (20640, 8)
y shape (target):   (20640,)

Feature columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


## 3. Numerical vs. Categorical Features

Two broad kinds of features show up in tabular data:

- **Numerical features** are measured quantities — they have a
  meaningful order and magnitude, and arithmetic on them makes sense
  (e.g. "$50,000 income" is twice "$25,000 income"). Examples here:
  `MedInc`, `HouseAge`, `Population`, `Latitude`.
- **Categorical features** represent a label or category from a
  fixed set of options, with no inherent numeric meaning (e.g.
  `"Near Ocean"` isn't "more" or "less" than `"Inland"` — you can't
  average two categories together). A model can't use raw text
  categories directly, so they need to be **encoded** into numbers
  first — which is exactly what One-Hot Encoding and Label Encoding
  (demonstrated next) are for.

Let's check what we actually have in this dataset:


In [ ]:
housing_df.dtypes


MedInc         float64
HouseAge       float64
AveRooms       float64
AveBedrms      float64
Population     float64
AveOccup       float64
Latitude       float64
Longitude      float64
MedHouseVal    float64
dtype: object

Every single column — all 8 features and the target — is `float64`.
There are no categorical (text/object) columns in this dataset at
all, which is exactly why the encoding demos below use a small,
separately-constructed example instead of a column from
`housing_df`.


## 4. One-Hot Encoding (Demonstrated on a Small Example)

Since `housing_df` has no categorical columns, we'll illustrate
One-Hot Encoding on a small made-up example feature —
`ocean_proximity_example` — similar in spirit to a column some other
versions of California housing data include, but **not part of this
dataset or this project's real features**. One-Hot Encoding creates
one new binary (0/1) column per category, with no column implying
any category is "bigger" than another.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

example_categories = pd.DataFrame({
    "ocean_proximity_example": [
        "Near Bay", "Inland", "Near Ocean", "Inland", "Near Bay", "Island"
    ]
})
print("Example categorical feature:")
print(example_categories)

one_hot_encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = one_hot_encoder.fit_transform(example_categories[["ocean_proximity_example"]])

one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=one_hot_encoder.get_feature_names_out(["ocean_proximity_example"]),
)
print("\nOne-Hot Encoded result:")
one_hot_df


Example categorical feature:
  ocean_proximity_example
0                Near Bay
1                  Inland
2              Near Ocean
3                  Inland
4                Near Bay
5                  Island

One-Hot Encoded result:


,ocean_proximity_example_Inland,ocean_proximity_example_Island,ocean_proximity_example_Near Bay,ocean_proximity_example_Near Ocean
0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0
5,0.0,1.0,0.0,0.0


## 5. Label Encoding (Demonstrated on the Same Example)

Label Encoding instead assigns each category a single integer
(0, 1, 2, ...) in one column. It's more compact than One-Hot
Encoding, but it silently implies an ORDER between categories (e.g.
`"Inland"=0`, `"Island"=1`, `"Near Bay"=2` — the model may treat
`"Near Bay"` as "greater than" `"Inland"`, which isn't meaningful
here). For that reason, `LabelEncoder` is technically intended by
scikit-learn for encoding TARGET labels, not input features;
`OrdinalEncoder` is the feature-oriented equivalent. We show
`LabelEncoder` here simply because it was asked for by name.


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded = label_encoder.fit_transform(example_categories["ocean_proximity_example"])

label_encoded_df = example_categories.copy()
label_encoded_df["ocean_proximity_label_encoded"] = label_encoded

print("Category -> integer mapping:")
for category, code_value in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  {category!r} -> {code_value}")

label_encoded_df


Category -> integer mapping:
  'Inland' -> 0
  'Island' -> 1
  'Near Bay' -> 2
  'Near Ocean' -> 3


,ocean_proximity_example,ocean_proximity_label_encoded
0,Near Bay,2
1,Inland,0
2,Near Ocean,3
3,Inland,0
4,Near Bay,2
5,Island,1


## 6. Why California Housing Doesn't Need Categorical Encoding

As shown in Section 3, all 8 input features (`MedInc`, `HouseAge`,
`AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`,
`Longitude`) are already numeric (`float64`) — there is no
text/category column to encode. `Latitude` and `Longitude` do
represent *location*, but they're expressed as continuous numeric
coordinates rather than named regions, so they behave like ordinary
numerical features (usable directly, or optionally scaled) rather
than categories needing One-Hot/Label Encoding. That's why
`src/preprocessing.py` in this project only imputes, caps outliers,
and scales — it never encodes anything.


## 7. Feature Scaling with StandardScaler

Our numeric features are on very different scales — e.g. `Population`
is in the thousands while `AveBedrms` is close to 1. Many models
(and especially anything regularized, like Ridge) perform better
when features are on a comparable scale. `StandardScaler` transforms
each column to have mean 0 and standard deviation 1:


In [ ]:
from sklearn.preprocessing import StandardScaler

demo_scaler = StandardScaler()
X_scaled_demo = demo_scaler.fit_transform(X)

print("Before scaling (first row):")
print(X.iloc[0])
print("\nAfter scaling (first row):")
print(pd.Series(X_scaled_demo[0], index=X.columns))

print("\nMean of each scaled column (should be ~0):")
print(np.round(X_scaled_demo.mean(axis=0), 8))
print("\nStd of each scaled column (should be ~1):")
print(np.round(X_scaled_demo.std(axis=0), 8))


Before scaling (first row):
MedInc          8.325200
HouseAge       41.000000
AveRooms        6.984127
AveBedrms       1.023810
Population    322.000000
AveOccup        2.555556
Latitude       37.880000
Longitude    -122.230000
Name: 0, dtype: float64

After scaling (first row):
MedInc        2.344766
HouseAge      0.982143
AveRooms      0.628559
AveBedrms    -0.153758
Population   -0.974429
AveOccup     -0.049597
Latitude      1.052548
Longitude    -1.327835
dtype: float64

Mean of each scaled column (should be ~0):
[-0.  0.  0.  0. -0. -0. -0. -0.]

Std of each scaled column (should be ~1):
[1. 1. 1. 1. 1. 1. 1. 1.]


*(Note: this demo scaler is fit on the full `X` purely to show what
scaling itself does. Section 10 below shows the CORRECT way to do
this for a real train/test workflow — fitting only on `X_train`.)*


## 8. Feature Selection by Correlation with MedHouseVal

Not every feature is equally useful for predicting price. One simple,
common feature-selection technique is to rank features by their
correlation with the target and keep the strongest ones.


In [ ]:
correlation_with_target = X.corrwith(y).sort_values(key=lambda s: s.abs(), ascending=False)
print("Correlation of each feature with MedHouseVal (sorted by strength):")
print(correlation_with_target)


Correlation of each feature with MedHouseVal (sorted by strength):
MedInc        0.688075
AveRooms      0.151948
Latitude     -0.144160
HouseAge      0.105623
AveBedrms    -0.046701
Longitude    -0.045967
Population   -0.024650
AveOccup     -0.023737
dtype: float64


In [ ]:
CORRELATION_THRESHOLD = 0.1
selected_features = correlation_with_target[correlation_with_target.abs() > CORRELATION_THRESHOLD].index.tolist()

print(f"Features selected (|correlation| > {CORRELATION_THRESHOLD}):")
print(selected_features)
print(f"\nSelected {len(selected_features)} out of {X.shape[1]} original features.")

X_selected = X[selected_features]
X_selected.head()


Features selected (|correlation| > 0.1):
['MedInc', 'AveRooms', 'Latitude', 'HouseAge']

Selected 4 out of 8 original features.


,MedInc,AveRooms,Latitude,HouseAge
0,8.3252,6.984127,37.88,41.0
1,8.3014,6.238137,37.86,21.0
2,7.2574,8.288136,37.85,52.0
3,5.6431,5.817352,37.85,52.0
4,3.8462,6.281853,37.85,52.0


## 9. Data Leakage: Fit on Train, Transform on Test

**Data leakage** happens when information from outside the training
set (most commonly, the test set) influences how a model or a
preprocessing step is fit. For a transformer like `StandardScaler`,
that means: the mean and standard deviation it learns must come from
`X_train` ONLY.

If we instead fit the scaler on the *full* dataset (train + test
combined) before splitting, the learned mean/std would be nudged by
test-set values — meaning the "training" data has secretly already
"seen" a statistical summary of the test set. That inflates how good
the model looks during evaluation, in a way that won't hold up on
truly new, unseen data.

The correct pattern:
1. Split into train/test FIRST.
2. `scaler.fit(X_train)` — learn mean/std from training data only.
3. `scaler.transform(X_train)` AND `scaler.transform(X_test)` — apply
   those same learned statistics to both sets. Never call `.fit()`
   or `.fit_transform()` on `X_test`.


## 10. Train/Test Split, Then Fit StandardScaler ONLY on X_train

Let's demonstrate the correct pattern directly, and prove the point
made in Section 9 by comparing the training-set mean to the
full-dataset mean.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)


X_train shape: (16512, 8)
X_test shape:  (4128, 8)


In [ ]:
# Fit ONLY on X_train.
train_scaler = StandardScaler()
train_scaler.fit(X_train)

# Apply the SAME fitted scaler to both train and test -- transform only.
X_train_scaled = train_scaler.transform(X_train)
X_test_scaled = train_scaler.transform(X_test)

print("Scaler mean_, learned from X_train only:")
print(pd.Series(train_scaler.mean_, index=X_train.columns))

print("\nFor comparison, the mean of the FULL dataset (train + test combined):")
print(X.mean())

print("\nThese are slightly different -- if we'd fit on the full X instead")
print("of X_train alone, we would have leaked a bit of test-set information")
print("into the statistics used to transform the training data.")


Scaler mean_, learned from X_train only:
MedInc           3.880754
HouseAge        28.608285
AveRooms         5.435235
AveBedrms        1.096685
Population    1426.453004
AveOccup         3.096961
Latitude        35.643149
Longitude     -119.582290
dtype: float64

For comparison, the mean of the FULL dataset (train + test combined):
MedInc           3.870671
HouseAge        28.639486
AveRooms         5.429000
AveBedrms        1.096675
Population    1425.476744
AveOccup         3.070655
Latitude        35.631861
Longitude     -119.569704
dtype: float64

These are slightly different -- if we'd fit on the full X instead
of X_train alone, we would have leaked a bit of test-set information
into the statistics used to transform the training data.


## 11. Shapes Before and After Transformation

A quick summary of every shape involved in the split + scale
pipeline above.


In [ ]:
shape_summary = pd.DataFrame(
    {
        "Object": ["X (full)", "X_train (raw)", "X_test (raw)",
                    "X_train_scaled", "X_test_scaled"],
        "Shape": [X.shape, X_train.shape, X_test.shape,
                   X_train_scaled.shape, X_test_scaled.shape],
    }
)
shape_summary


,Object,Shape
0,X (full),"(20640, 8)"
1,X_train (raw),"(16512, 8)"
2,X_test (raw),"(4128, 8)"
3,X_train_scaled,"(16512, 8)"
4,X_test_scaled,"(4128, 8)"


## 12. Feature Engineering Conclusions

- **No categorical encoding needed for this dataset.** All 8 input
  features are numeric (`float64`); the One-Hot and Label Encoding
  demos above used a separately-constructed example feature purely
  to illustrate the techniques, since this project has no actual use
  for them on this data.
- **Scaling matters here.** The raw features span wildly different
  ranges (`Population` in the thousands vs. `AveBedrms` near 1), so
  `StandardScaler` is a meaningful preprocessing step before feeding
  these features into scale-sensitive models like Ridge regression.
- **Feature selection by correlation identified 4 standout features.**
  Using a `|correlation| > 0.1` threshold against `MedHouseVal`
  selected `MedInc`, `AveRooms`, `Latitude`, and `HouseAge` out of
  the 8 available — `MedInc` alone is far ahead of the others at a
  correlation of about **0.69**, while the other three are only
  weakly correlated (between about 0.10 and 0.15 in absolute value).
  The remaining four features (`AveBedrms`, `Longitude`, `Population`,
  `AveOccup`) all fall below the threshold, each with |correlation|
  under 0.05.
- **Data leakage is a real, measurable risk, not just a theoretical
  one.** Section 10 showed the `X_train`-only mean differs from the
  full-dataset mean for every single feature (e.g. `MedInc`: about
  3.881 from `X_train` vs. about 3.871 from the full data) — a small
  difference, but a concrete demonstration of exactly the kind of
  leakage that fitting a transformer on the wrong data would
  introduce.
- **The correct, leakage-safe pattern demonstrated here** — split
  first, `fit()` only on `X_train`, `transform()` on both `X_train`
  and `X_test` — is the same pattern this project's own
  `src/preprocessing.py` module follows for the real pipeline used in
  training and evaluation.
